In [5]:
import json
from tqdm import tqdm
from openai import OpenAI

In [ ]:
import json
from openai import OpenAI

def load_jsonl(jsonl_path):
    """
    Loads a line-delimited JSON (.jsonl) file into a list of dicts.
    """
    data = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:  # Skip empty lines
                data.append(json.loads(line))
    return data

def load_json(json_path):
    """
    Loads a standard JSON file (containing a list or dict) into a Python object.
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        return json.load(f)

# 1. Load the .jsonl file
jsonl_data = load_jsonl("../results/meditron_responses.jsonl")

# 2. Load the .json file
json_data = load_json("../results/parsed_prompts_tasks_x_topics.json")

# Initialize your client
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key="TO ADD"
)

json_data_by_id = {item['id']: item for item in json_data}

# This list will hold all final results
results_list = []

for i, item in enumerate(tqdm(jsonl_data)): # stopped at 252-7 ->iteration 2528
    if i > 2528:
        prompt_id = item.get('prompt_id')
        seq_num = item.get('sequence_number')
        response_text = item.get('response')

        match = json_data_by_id.get(prompt_id)
        if match:
            # Call the completion API
            completion = client.chat.completions.create(
                model="nvidia/llama-3.1-nemotron-70b-reward",
                messages=[
                    {"role": "user", "content": match["prompt"]},
                    {"role": "assistant", "content": response_text}
                ]
            )
            reward = completion.choices[0].message.content
            
            # Build the result dictionary
            # ID is a combination of prompt_id and sequence_number
            result_dict = {
                "id": f"{prompt_id}-{seq_num}",
                "messages": [
                    {"role": "user", "content": match["prompt"]},
                    {"role": "assistant", "content": response_text}
                ],
                "reward": reward[7:]
            }
            
            # Append it to our results list
            results_list.append(result_dict)

# 3. Write all results to a new JSON file
output_path = "../results/DPO_test_dataset.json"
with open(output_path, "w", encoding="utf-8") as outfile:
    # Use indent=2 for readable formatting; ensure_ascii=False for UTF-8
    json.dump(results_list, outfile, ensure_ascii=False, indent=2)

print(f"Results written to {output_path}")

100%|██████████| 6370/6370 [24:23<00:00,  4.35it/s]  


Results written to ../results/DPO_test_dataset.json


In [18]:
import json

def load_json(json_path):
    """
    Loads a JSON file into a Python object.
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def concatenate_json_files(json_paths, output_path):
    """
    Loads multiple JSON files, concatenates their content, and writes the combined result to a new JSON file.
    """
    concatenated_data = []
    
    for path in json_paths:
        # Load the current JSON file
        data = load_json(path)
        # Extend the concatenated data with the current file's entries
        concatenated_data.extend(data)
    
    # Write the concatenated data to the output file
    with open(output_path, 'w', encoding='utf-8') as outfile:
        json.dump(concatenated_data, outfile, ensure_ascii=False, indent=2)

    print(f"Combined data written to {output_path}")

# Paths to the input JSON files
json_file_paths = [
    "../results/DPO_test_dataset_252-7.json",  # Replace with the actual path to the first JSON file
    "../results/DPO_test_dataset_636-9.json"  # Replace with the actual path to the second JSON file
]

# Path to the output JSON file
output_file_path = "../results/DPO_test_dataset.json"

# Concatenate the JSON files
concatenate_json_files(json_file_paths, output_file_path)


Combined data written to ../results/DPO_test_dataset.json
